In [1]:
import os
import numpy as np
import wfdb
import scipy.signal as signal
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


In [2]:
def bandpass_filter(signal_data, fs=100, lowcut=0.5, highcut=40, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, signal_data)

def notch_filter(signal_data, fs=100, freq=50.0, Q=30.0):
    nyq = 0.5 * fs
    w0 = freq / nyq
    b, a = signal.iirnotch(w0, Q)
    return signal.filtfilt(b, a, signal_data)

def preprocess_signal(sig, fs=100):
    sig = np.asarray(sig, dtype=np.float32)
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)

    sig = bandpass_filter(sig, fs)
    sig = notch_filter(sig, fs)

    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    std = np.std(sig)
    if std < 1e-8:
        return np.zeros_like(sig, dtype=np.float32)

    sig = (sig - np.mean(sig)) / (std + 1e-8)
    return sig.astype(np.float32)


In [3]:
def segment_signal(signal_data, annotations, fs=100):
    segments = []
    labels = []
    segment_length = fs * 60

    for i, label in enumerate(annotations):
        start = i * segment_length
        end = start + segment_length

        if end <= len(signal_data):
            seg = signal_data[start:end]
            if not np.isfinite(seg).all():
                continue
            segments.append(seg)
            labels.append(1 if label == 'A' else 0)

    return np.array(segments, dtype=np.float32), np.array(labels, dtype=np.int64)


In [4]:
def load_apnea_dataset(data_path):
    X_all = []
    y_all = []
    skipped = []

    # use records that have apnea annotations
    records = sorted(set(f.split('.')[0] for f in os.listdir(data_path) if f.endswith('.apn')))

    for idx, rec in enumerate(records, 1):
        try:
            record = wfdb.rdrecord(os.path.join(data_path, rec))
            annotation = wfdb.rdann(os.path.join(data_path, rec), 'apn')

            signal_data = record.p_signal[:, 0]
            signal_data = preprocess_signal(signal_data, fs=int(record.fs))

            segments, labels = segment_signal(signal_data, annotation.symbol, fs=int(record.fs))
            if len(segments) == 0:
                skipped.append((rec, 'no valid segments'))
                continue

            X_all.append(segments)
            y_all.append(labels)

            if idx % 10 == 0:
                print(f"Loaded {idx}/{len(records)} records")

        except Exception as e:
            skipped.append((rec, str(e)))

    if not X_all:
        raise ValueError('No valid records were loaded.')

    X_all = np.concatenate(X_all, axis=0).astype(np.float32)
    y_all = np.concatenate(y_all, axis=0).astype(np.int64)

    finite_ratio = float(np.isfinite(X_all).mean())
    class_counts = np.bincount(y_all, minlength=2)
    print(f"Total segments: {len(X_all)} | finite_ratio={finite_ratio:.6f}")
    print(f"Class counts -> Normal(0): {class_counts[0]}, Apnea(1): {class_counts[1]}")
    if skipped:
        print(f"Skipped records: {len(skipped)}")

    return X_all, y_all


In [5]:
data_path = "dataset/1.0.0"

X, y = load_apnea_dataset(data_path)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")
train_counts = np.bincount(y_train, minlength=2)
test_counts = np.bincount(y_test, minlength=2)
print(f"Train class counts -> 0: {train_counts[0]}, 1: {train_counts[1]}")
print(f"Test class counts  -> 0: {test_counts[0]}, 1: {test_counts[1]}")


Loaded 10/86 records
Loaded 20/86 records
Loaded 50/86 records
Loaded 60/86 records
Loaded 70/86 records
Loaded 80/86 records
Total segments: 38222 | finite_ratio=1.000000
Class counts -> Normal(0): 23555, Apnea(1): 14667
Skipped records: 8
Train samples: 30577, Test samples: 7645
Train class counts -> 0: 18844, 1: 11733
Test class counts  -> 0: 4711, 1: 2934


In [6]:
class ApneaDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(ApneaDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(ApneaDataset(X_test, y_test), batch_size=32)


In [7]:
class PCSA(nn.Module):
    def __init__(self, channels):
        super(PCSA, self).__init__()

        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(channels, channels // 4)
        self.fc2 = nn.Linear(channels // 4, channels)

        self.conv_spatial = nn.Conv1d(channels, channels, kernel_size=7, padding=3)

    def forward(self, x):
        b, c, l = x.size()

        # Channel attention
        y = self.avg_pool(x).view(b, c)
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1)
        x = x * y

        # Spatial attention
        s = torch.sigmoid(self.conv_spatial(x))
        x = x * s

        return x

In [8]:
class ParallelConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

    def forward(self, x):
        return self.block(x)


class CustomApneaModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = ParallelConvBlock(1, 32, 3)
        self.conv2 = ParallelConvBlock(1, 32, 5)
        self.conv3 = ParallelConvBlock(1, 32, 7)

        self.pcsa = PCSA(96)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc1 = nn.Linear(96, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x)
        x3 = self.conv3(x)

        x = torch.cat([x1, x2, x3], dim=1)

        x = self.pcsa(x)
        x = self.global_pool(x).squeeze(-1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomApneaModel().to(device)

class_counts = np.bincount(y_train, minlength=2).astype(np.float32)
class_weights = class_counts.sum() / (2.0 * (class_counts + 1e-8))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Device: {device}")
print(f"Class weights used in loss: {class_weights}")


Device: cuda
Class weights used in loss: [0.81131923 1.3030342 ]


In [10]:
from sklearn.metrics import confusion_matrix, accuracy_score

def train(model, loader):
    model.train()
    total_loss = 0.0
    valid_batches = 0
    skipped_batches = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        if not torch.isfinite(X_batch).all():
            skipped_batches += 1
            continue

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        if not torch.isfinite(loss):
            skipped_batches += 1
            continue

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        valid_batches += 1

    mean_loss = total_loss / max(valid_batches, 1)
    return mean_loss, valid_batches, skipped_batches


def evaluate(model, loader):
    model.eval()
    preds = []
    truths = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            predicted = torch.argmax(outputs, dim=1)

            preds.extend(predicted.cpu().numpy())
            truths.extend(y_batch.numpy())

    preds = np.array(preds)
    truths = np.array(truths)

    tn, fp, fn, tp = confusion_matrix(truths, preds, labels=[0, 1]).ravel()

    accuracy = accuracy_score(truths, preds)
    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    pos_pred_rate = (preds == 1).mean()

    return accuracy, sensitivity, specificity, pos_pred_rate


In [11]:
for epoch in range(20):
    loss, valid_batches, skipped_batches = train(model, train_loader)
    acc, sen, spec, pos_rate = evaluate(model, test_loader)

    print(f"Epoch {epoch+1}")
    print(f"Loss: {loss:.4f} | valid_batches: {valid_batches} | skipped_batches: {skipped_batches}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Sensitivity: {sen:.4f}")
    print(f"Specificity: {spec:.4f}")
    print(f"Positive prediction rate: {pos_rate:.4f}")
    print("-" * 40)


Epoch 1
Loss: 0.5798 | valid_batches: 956 | skipped_batches: 0
Accuracy: 0.7717
Sensitivity: 0.8722
Specificity: 0.7092
Positive prediction rate: 0.5139
----------------------------------------
Epoch 2
Loss: 0.4476 | valid_batches: 956 | skipped_batches: 0
Accuracy: 0.8038
Sensitivity: 0.8190
Specificity: 0.7943
Positive prediction rate: 0.4411
----------------------------------------
Epoch 3
Loss: 0.4121 | valid_batches: 956 | skipped_batches: 0
Accuracy: 0.7881
Sensitivity: 0.9185
Specificity: 0.7069
Positive prediction rate: 0.5332
----------------------------------------
Epoch 4
Loss: 0.3929 | valid_batches: 956 | skipped_batches: 0
Accuracy: 0.8069
Sensitivity: 0.9271
Specificity: 0.7321
Positive prediction rate: 0.5209
----------------------------------------
Epoch 5
Loss: 0.3811 | valid_batches: 956 | skipped_batches: 0
Accuracy: 0.8246
Sensitivity: 0.9219
Specificity: 0.7640
Positive prediction rate: 0.4993
----------------------------------------
Epoch 6
Loss: 0.3680 | valid_b

In [12]:
# Final evaluation on the test set
final_acc, final_sen, final_spec, final_pos_rate = evaluate(model, test_loader)

print("\nFinal Test Metrics")
print(f"Accuracy   : {final_acc:.4f}")
print(f"Sensitivity: {final_sen:.4f}")
print(f"Specificity: {final_spec:.4f}")
print(f"Pos. Rate  : {final_pos_rate:.4f}")



Final Test Metrics
Accuracy   : 0.8623
Sensitivity: 0.9400
Specificity: 0.8138
Pos. Rate  : 0.4755
